# Regressione delta_skill_neve vs delta_cover

Quarto pezzo del quadro di convergenza discusso (vedi 03, 04): oltre a
`delta_cover` vs `delta_skill_tas` e `delta_cover` vs `delta_skill_albedo`,
si aggiunge `delta_cover` vs `delta_skill_neve`.

Come l'albedo (notebook 04), la neve e' **libera/prognostica in entrambi gli
esperimenti** (nessuna forzatura con osservazioni, a differenza della cover) e
ha un'osservazione indipendente vera (ERA5): `delta_skill_neve` e' un
miglioramento di skill genuino sia per SENS sia per CTRL, nessun disegno
ibrido necessario.

```
X = delta_cover        = cover_SENS - cover_CTRL (anomalia, non circolare)
Y = delta_skill_neve    = skill_neve_SENS - skill_neve_CTRL (vs ERA5, genuino)
```

Variabile di neve: `snd` (snow depth) di default — vedi `snow_var` sotto,
cambiare qui se serve passare a `sd`. Stesse tre mappe (Pearson slope,
Pearson r, Spearman rho) + scatter sul box Siberia di 03/04, stessa maschera
a soglia fissa `1e-3` su `delta_cover`.


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
variables = ['cvh', 'cvl']
snow_var = 'snd'  # 'sd' dovrebbe essere equivalente, cambiare qui se serve
SAVE_PATH = str(FIG_DIR)


In [ ]:
# La logica di calcolo sta in cover_tas_lib.py (stesso modulo di 01/02/03/04,
# nuova funzione run_one_cover_snow). Processi spawn freschi, stesso pattern
# robusto adottato in questa sessione.
sys.path.insert(0, os.getcwd())
from cover_tas_lib import run_one_cover_snow, LEADS


In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, y1, y2, SAVE_PATH, snow_var) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_cover_snow, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
